In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/Users/syedjunaid3786@gmail.com/consolidated_pipeline/setup_1/utilities

In [0]:
print(bronze_schema, silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ete", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = f"s3a://S3 Path/{data_source}.csv"
print(base_path)

In [0]:
df_bronze = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    
    .withColumn("read_timestamp", f.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)
display(df_bronze.limit(10))

In [0]:
df_bronze.write \
    .format("delta") \
    .option("delta.enablechangeDataFeed","true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

print(f" Successfully written to {catalog}.{bronze_schema}.{data_source}")

### Silver Processing..


In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter("count > 1")

display(df_duplicates)

In [0]:
print("total rows before duplicates are dropped",df_bronze.count())
df_silver=df_bronze.dropDuplicates(["customer_id"])
print("total rows after duplicates are dropped",df_silver.count())



In [0]:
display(
    df_silver.filter(f.col("customer_name")!= f.trim(f.col("customer_name")))
)

In [0]:
df_silver =df_silver.withColumn("customer_name",f.trim(f.col("customer_name")))
df_silver = df_silver.withColumn("customer_name", f.initcap(f.col("customer_name")))

In [0]:
print("Cities before cleaning:")
df_silver.select("city").distinct().show()

In [0]:
df_silver = df_silver.withColumn("city", f.initcap(f.col("city")))

df_silver = df_silver.withColumn(
    "city", f.when(f.col("city") == "Bostn", "Boston").otherwise(f.col("city"))
)

print("Cities after cleaning:")
df_silver.select("city").distinct().show()

In [0]:
df_silver.filter(f.col("city").isNull()).show(truncate=False)

In [0]:
df_silver = df_silver.withColumn(
    "city", 
    f.when(f.col("city") == "Unknown", None).otherwise(f.col("city")))

windowSpec = Window.partitionBy("customer_name")

df_silver = df_silver.withColumn(
    "city", 
    f.first("city", ignorenulls=True).over(windowSpec)
)

df_silver = df_silver.fillna({"city": "Unknown"})

print("Cities backfilled using matching customer names:")
display(df_silver.limit(5))    

In [0]:
df_silver = (
    df_silver.withColumn(
        "customer", 
        f.concat_ws("-", f.col("customer_name"), f.col("city"))
    )
)

df_silver = (
    df_silver
    .withColumn("market", f.lit("USA"))
    .withColumn("platform", f.lit("PC Accessories Hub"))
    .withColumn("channel", f.lit("Acquisition"))
)

print("Conformed Silver Data:")
display(df_silver.limit(5))

In [0]:
silver_table_name = f"{catalog}.{silver_schema}.{data_source}"

df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(silver_table_name)

print(f"Cleaned and conformed data successfully written to {silver_table_name}")

### Gold Processing...

In [0]:
df_silver = spark.table(f"{catalog}.{silver_schema}.{data_source}")

# Selecting only the columns needed for business reporting
df_gold = df_silver.select(
    "customer_id", 
    "customer_name", 
    "city", 
    "customer", 
    "market", 
    "platform", 
    "channel"
)

child_gold_table = f"{catalog}.{gold_schema}.sb_dim_{data_source}"

df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(child_gold_table)

print(f"Child Gold table saved to: {child_gold_table}")

In [0]:
parent_table_name = f"{catalog}.{gold_schema}.dim_customers"
delta_table = DeltaTable.forName(spark, parent_table_name)

df_child_customers = spark.table(child_gold_table).select(
    f.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Child customers successfully merged into Parent database!")